In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

from sklearn.model_selection import train_test_split

from sklearn.metrics import *

import tensorflow as tf

import keras

import numpy as np

import matplotlib.pyplot as plt


from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout


# EfficientNetB0 Imports :

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from glob import glob


import os
import cv2 # computer vision cv2 images would be read


In [20]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [21]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

In [22]:
path = r"D:\Imarticus\Capstone Project_2\Psoriasis Disease Detection NN Model"

cate = ['Normal_Dry_Oil Skin' , 'Psoriasis Skin']

In [23]:
image_size = 224
input_image = []

for i in cate:
    folders = os.path.join(path, i)
    label = cate.index(i)     # to perform labeling of dog as : 0 and cat as : 1  

    for image in os.listdir(folders):
        image_path = os.path.join(folders , image)
        image_array = cv2.imread(image_path)

        if image_array is None:
            continue  # skip corrupted images
        image_array = cv2.resize(image_array , (image_size , image_size))   # resizing the image 
                                                                            # because in raw data each image size is different and we require 
                                                                            # same size
        input_image.append([image_array , label])

        
        

In [24]:
np.random.shuffle(input_image)

X = []
Y = []

for X_values , labels in input_image:
    X.append(X_values)
    Y.append(labels)



#  Separate X ----> pixels and Y ----> 0 and 1
#  Entire data is stacked on categories
#  1000 images ---> 500 dogs and 500 cats
#  

In [25]:
X = np.array(X)
Y = np.array(Y)

In [26]:
# X = X/255

In [27]:
X_train = X[0 : 7722]
Y_train = Y[0:7722]

X_test = X[7722 :]
Y_test = Y[7722 :]

In [28]:
X_test.shape

(2295, 224, 224, 3)

In [29]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)

X_test = np.array(X_test)

In [30]:
X_train.shape[1:]

(224, 224, 3)

In [31]:
Y_test[0]

np.int64(1)

In [32]:
# X = preprocess_input(X)

In [33]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)


for layer in base_model.layers:
    layer.trainable = False

                                        # Custom classifier on top of VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D

x = GlobalAveragePooling2D()(base_model.output)

x = Dense(256, activation='relu')(x)

x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)    # extra dense layer

output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


In [34]:
adam = tf.keras.optimizers.Adam(learning_rate=0.0001)

model.compile(
    optimizer=adam,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [35]:
history = model.fit(
    X_train,
    Y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=32
)


Epoch 1/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.9344 - loss: 0.1785 - val_accuracy: 0.9871 - val_loss: 0.0403
Epoch 2/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 598s 1s/step - accuracy: 0.9862 - loss: 0.0425 - val_accuracy: 0.9922 - val_loss: 0.0200
Epoch 3/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 326s 2s/step - accuracy: 0.9903 - loss: 0.0290 - val_accuracy: 0.9961 - val_loss: 0.0158
Epoch 4/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 316s 1s/step - accuracy: 0.9947 - loss: 0.0197 - val_accuracy: 0.9974 - val_loss: 0.0115
Epoch 5/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 305s 1s/step - accuracy: 0.9945 - loss: 0.0164 - val_accuracy: 0.9974 - val_loss: 0.0093
Epoch 6/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 269s 1s/step - accuracy: 0.9964 - loss: 0.0099 - val_accuracy: 0.9974 - val_loss: 0.0089
Epoch 7/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.9984 - loss: 0.0076 - val_accuracy: 0.9981 - val_loss: 0.0084
Epoch 8/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 302s 2s/step - accuracy: 0.9976 - loss: 0.0078 - val_accu

In [36]:
pred = model.predict(X_test)
pred_classes = pred.argmax(axis=1)

72/72 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step


In [37]:
print(confusion_matrix(Y_test , pred_classes)) 
print('')
print(classification_report(Y_test , pred_classes))

[[1153    4]
 [   0 1138]]

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1157
           1       1.00      1.00      1.00      1138

    accuracy                           1.00      2295
   macro avg       1.00      1.00      1.00      2295
weighted avg       1.00      1.00      1.00      2295

